# Day 3 · Exercise 6: When should a store reorder, and how much?

**What you'll be able to say by the end**
1. "An **(s, S) policy**: when stock on hand plus on order drops to **s**, order enough to bring it back up to **S**. s protects against running out; S − s sets how often you order."
2. "Set them with formulas: **s = demand over the lead time + safety stock**, and **S = s + EOQ**. Then check by simulation."
3. "**Cycle service level** (the chance a cycle has no stockout) and **fill rate** (the share of demand served) are different numbers. An 80% cycle service level already gave a 99.4% fill rate here."
4. "Here's the **cost vs fill-rate curve**. Going from 95% to 99% fill rate costs about $28 more a year for this item."
5. "If lead times vary, the safety stock needs a second term. Without it, fill rate fell from 99.3% to 97.5%."

**Difference from exercise 5:** the newsvendor orders once. Here, the store reorders over and over, all year.

_Setup check: make sure this notebook runs on the repo's `.venv`. Nothing to learn here._

In [1]:
import sys
from pathlib import Path

VENV = (Path.cwd().parent / ".venv").resolve()
assert Path(sys.prefix).resolve() == VENV, f"Wrong kernel: {sys.prefix}. Select 'supply-chain-or (.venv)'."
print("Kernel:", sys.executable)

Kernel: /Users/abhijeet/Personal/supply_chain_or/.venv/bin/python


## 1. The story

A store sells one household item, `HOUSEHOLD_1_517` from M5 store CA_1, about **8 units a day**, every day. Each day it checks its stock and can place an order with the DC. An order arrives **4 days** later (the **lead time**).

Two costs pull against each other:
- **Ordering cost, $10 per order:** admin, a truck slot, receiving. Order rarely, in big batches, to save this.
- **Holding cost, 25% of the unit's $10 cost per year** (about $0.007 per unit per day): money tied up, shelf space, damage. Keep stock low to save this.

And there's a **service target**: running out loses sales (unmet demand is lost, not back-ordered).

We **learn** the demand pattern from one year of real sales, then **test** on the next year, which the policy never sees in advance.

In [2]:
import math

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from scipy.stats import norm

ROOT = Path.cwd().parent  # notebooks/ is the working folder
OUT = ROOT / "outputs"
daily = pd.read_csv(ROOT / "data" / "processed" / "m5_ca1_daily.csv")

ITEM = "HOUSEHOLD_1_517"
series = daily[daily.item_id == ITEM].sort_values("date")
demand_all = series["units"].to_numpy()
history = demand_all[-728:-364]  # the year we learn from
test = demand_all[-364:]         # the year we simulate on (the policy never sees it in advance)

# Costs (assumptions, per unit of this item)
UNIT_COST = 10.0                       # $ we pay per unit
HOLDING_PER_DAY = 0.25 * UNIT_COST / 365  # 25% a year to carry stock: capital, space, damage
ORDER_COST = 10.0                      # $ per order: admin, truck slot, receiving
LEAD_TIME = 4                          # days from order to delivery
TARGET_CSL = 0.95                      # aim: no stockout in 95% of replenishment cycles

mu, sigma = history.mean(), history.std()
print(f"Learning year: mean {mu:.2f} units/day, standard deviation {sigma:.2f}")
print(f"Test year:     mean {test.mean():.2f} units/day, standard deviation {test.std():.2f}")

fig = go.Figure(go.Scatter(y=demand_all[-728:], mode="lines", line=dict(width=1),
                           hovertemplate="day %{x}<br>%{y} units<extra></extra>"))
fig.add_vrect(x0=0, x1=364, fillcolor="#4C78A8", opacity=0.08, annotation_text="learn from this year", annotation_position="top left")
fig.add_vrect(x0=364, x1=728, fillcolor="#F58518", opacity=0.08, annotation_text="test on this year", annotation_position="top left")
fig.update_layout(title=f"{ITEM}: daily sales, last two years", xaxis_title="day", yaxis_title="units sold",
                  height=380, width=1000, template="plotly_white", showlegend=False)
fig.show()

Learning year: mean 7.98 units/day, standard deviation 4.39
Test year:     mean 8.37 units/day, standard deviation 4.77


Daily sales jump around a lot (the standard deviation, 4.4, is more than half the mean, 8), but the **level is steady** across both years (8.0 vs 8.4 a day). That's what makes a fixed policy sensible. When the level drifts, you re-estimate it regularly.

## 2. The (s, S) policy, and a simulator to test it

**The rule, every day:**
1. Receive any delivery due today.
2. Sell what customers ask for, as far as stock lasts. Anything beyond that is a **lost sale**.
3. Work out the **inventory position** = stock on hand + stock already ordered but not yet arrived.
4. If the inventory position is **at or below s**, order enough to bring it **up to S**.

Why use the inventory position and not just what's on the shelf? Because an order that's already on its way will arrive. If you looked only at the shelf, you'd keep re-ordering every day while waiting for the truck.

**Two ways to score a policy:**
- **Fill rate** = units sold ÷ units customers asked for. *"What share of demand did we serve?"*
- **Cycle service level (CSL)** = the share of replenishment cycles (delivery to delivery) with **no** stockout at all. *"How often did we get through a cycle without running out?"*

The simulator below plays out a year day by day and tracks both, plus the costs. Read it top to bottom: it's the 4-step rule above, in code.

In [3]:
def simulate(s, S, demand, lead_time=LEAD_TIME, lead_time_sampler=None, seed=0):
    """Daily (s, S) policy with lost sales.

    Each day: receive deliveries due today, sell what we can (unmet demand is lost),
    then if inventory position (on hand + on order) <= s, order up to S.
    """
    rng = np.random.default_rng(seed)
    on_hand, pipeline = S, []  # pipeline: list of (arrival_day, quantity)
    held = sold = asked = orders = 0
    cycles = cycles_with_stockout = 0
    stockout_this_cycle = False

    for day, d in enumerate(demand):
        arriving = [q for t, q in pipeline if t == day]
        if arriving:
            on_hand += sum(arriving)
            pipeline = [(t, q) for t, q in pipeline if t != day]
            cycles += 1
            cycles_with_stockout += stockout_this_cycle
            stockout_this_cycle = False

        sale = min(on_hand, d)
        on_hand -= sale
        sold += sale
        asked += d
        stockout_this_cycle |= d > sale

        position = on_hand + sum(q for _, q in pipeline)
        if position <= s:
            lt = lead_time if lead_time_sampler is None else lead_time_sampler(rng)
            pipeline.append((day + lt, S - position))
            orders += 1
        held += on_hand

    holding, ordering = HOLDING_PER_DAY * held, ORDER_COST * orders
    return {"fill rate": sold / asked, "cycle service level": 1 - cycles_with_stockout / max(cycles, 1),
            "avg on hand": held / len(demand), "orders": orders,
            "holding $": holding, "ordering $": ordering, "total $": holding + ordering}

## 3. Set s and S with formulas

**s, the reorder point:** after you order, you wait 4 days. You need enough stock to cover demand during that wait:

> **s = average demand over the lead time + safety stock**

Average demand over 4 days is 4 × 8 = 32 units. But demand varies, so we add a buffer, the **safety stock**:

```latex
SS = z \, \sigma_d \sqrt{L}
```

- σ_d (sigma) is the daily standard deviation. Over L days, the swings partly cancel out, so the spread grows with √L, not L.
- z sets how cautious we are. To have no stockout in 95% of cycles (CSL = 95%), use z = 1.645, the 95th percentile of the normal curve. That's the newsvendor idea again: we pick a quantile of lead-time demand.

**S − s, how much to order each time:** the **economic order quantity (EOQ)** balances ordering cost against holding cost:

```latex
EOQ = \sqrt{\frac{2\,D\,K}{h}}
```

Here D = demand per day, K = $10 per order, and h = holding cost per unit per day. Big orders mean few order costs but lots of stock; small orders mean the reverse. EOQ is where the two balance.

In [4]:
def formula_policy(mu, sigma, lead_mean=LEAD_TIME, lead_sd=0.0, csl=TARGET_CSL):
    """s = expected demand over the lead time + safety stock; S = s + EOQ."""
    z = norm.ppf(csl)
    safety_stock = z * math.sqrt(lead_mean * sigma ** 2 + mu ** 2 * lead_sd ** 2)
    eoq = math.sqrt(2 * mu * ORDER_COST / HOLDING_PER_DAY)
    s = mu * lead_mean + safety_stock
    return {"z": z, "safety stock": safety_stock, "EOQ": eoq, "s": round(s), "S": round(s + eoq)}


policy = formula_policy(mu, sigma)
print(f"z = {policy['z']:.3f}")
print(f"safety stock = {policy['z']:.3f} x {sigma:.2f} x sqrt({LEAD_TIME}) = {policy['safety stock']:.1f} units")
print(f"s = {mu:.2f} x {LEAD_TIME} + {policy['safety stock']:.1f} = {policy['s']}")
print(f"EOQ = sqrt(2 x {mu:.2f} x {ORDER_COST:.0f} / {HOLDING_PER_DAY:.5f}) = {policy['EOQ']:.0f} units (about {policy['EOQ'] / mu:.0f} days of demand)")
print(f"S = s + EOQ = {policy['S']}")

z = 1.645
safety stock = 1.645 x 4.39 x sqrt(4) = 14.4 units
s = 7.98 x 4 + 14.4 = 46
EOQ = sqrt(2 x 7.98 x 10 / 0.00685) = 153 units (about 19 days of demand)
S = s + EOQ = 199


## 4. Does it work? Simulate the test year

Run the formula policy (s = 46, S = 199) on the real test year:

In [5]:
result = simulate(policy["s"], policy["S"], test)
pd.Series(result).round(3)

fill rate                1.000
cycle service level      1.000
avg on hand             94.376
orders                  19.000
holding $              235.295
ordering $             190.000
total $                425.295
dtype: float64

A clean year: **no stockouts at all** (fill rate and CSL both 100%), with 19 orders. But one year contains only 19 cycles. With a 95% target we'd expect about 1 bad cycle, so seeing 0 could just be luck.

**To get a fair score, simulate many plausible years.** We build each one by drawing 52 whole weeks at random from the test year. Using whole weeks keeps the weekday pattern intact. Then we average the results over 200 such years. We do this for four different CSL targets:

In [6]:
def resampled_year(seed, weeks=test.reshape(-1, 7)):
    """A plausible alternative year: 52 whole weeks drawn at random (with replacement) from the test year."""
    rng = np.random.default_rng(seed)
    return weeks[rng.integers(0, len(weeks), len(weeks))].ravel()


def average_over_years(s, S, n_years, **kwargs):
    runs = [simulate(s, S, resampled_year(k), seed=k, **kwargs) for k in range(n_years)]
    return pd.DataFrame(runs).mean().to_dict()


rows = []
for csl in [0.80, 0.90, 0.95, 0.99]:
    pol = formula_policy(mu, sigma, csl=csl)
    avg = average_over_years(pol["s"], pol["S"], 200)
    rows.append({"target CSL": csl, "s": pol["s"], "S": pol["S"], "achieved CSL": avg["cycle service level"],
                 "fill rate": avg["fill rate"], "total $": avg["total $"]})
service = pd.DataFrame(rows).set_index("target CSL")

fig = go.Figure()
fig.add_trace(go.Bar(x=[f"{c:.0%}" for c in service.index], y=100 * service["achieved CSL"], name="cycle service level",
                     hovertemplate="target %{x}<br>CSL %{y:.1f}%<extra></extra>"))
fig.add_trace(go.Bar(x=[f"{c:.0%}" for c in service.index], y=100 * service["fill rate"], name="fill rate",
                     hovertemplate="target %{x}<br>fill rate %{y:.1f}%<extra></extra>"))
fig.update_layout(title="Same policies, two ways to score them (average of 200 years)", barmode="group",
                  xaxis_title="target cycle service level used to set s", yaxis_title="%", yaxis_range=[70, 100.5],
                  height=430, width=900, template="plotly_white", legend=dict(orientation="h", y=1.1))
fig.show()
service.round(3)

,s,S,achieved CSL,fill rate,total $
target CSL,,,,,
0.80,39,192,0.825,0.994,403.995
0.90,43,196,0.919,0.998,414.043
0.95,46,199,0.954,0.999,421.415
0.99,52,205,0.986,1.000,436.190


**Two lessons in one chart:**

1. **The formula hits its target.** Aim for 80% CSL and get 82.5%; aim for 95% and get 95.4%; aim for 99% and get 98.6%. The safety stock formula works when its assumptions roughly hold (a steady level, a known lead time).

2. **CSL and fill rate are very different numbers.** At an 80% CSL, one cycle in five has a stockout, yet the **fill rate is 99.4%**. Why? Each order covers about 19 days. A stockout can only happen in the last day or so before the next delivery, and only a few units go unserved. "Ran out at some point" (CSL) is far more common than "lots of demand went unserved" (fill rate).

**Which does a planner care about?** Usually **fill rate**: it's closer to lost sales and revenue. CSL is easier to compute (it's just a quantile), so it's what the safety stock formula uses. **Always ask which one a "95% service level" target means.** Confusing them can mean carrying far more safety stock than needed.

## 5. Search for the best (s, S): the cost vs service curve

The formulas give a sensible starting policy. But we can just **try many (s, S) pairs** and see what each costs and how well it serves. Each policy below is averaged over 20 resampled years.

In [7]:
def frontier(grid):
    """Cheapest policy at or above each fill rate: the efficient frontier."""
    g = grid.sort_values("fill rate", ascending=False)
    g = g[g["total $"] == g["total $"].cummin()]
    return g.sort_values("fill rate")


grid = pd.DataFrame([{"s": s, "S": S, **average_over_years(s, S, 20)}
                     for s in range(10, 71, 4) for S in range(60, 361, 20) if S > s])
front = frontier(grid)
formula_avg = average_over_years(policy["s"], policy["S"], 20)
print(f"{len(grid)} policies simulated")

fig = go.Figure()
fig.add_trace(go.Scatter(x=100 * grid["fill rate"], y=grid["total $"], mode="markers", name="every (s, S) tried",
                         marker=dict(size=5, color="#BAB0AC"), customdata=grid[["s", "S"]],
                         hovertemplate="s=%{customdata[0]}, S=%{customdata[1]}<br>fill %{x:.1f}%<br>$%{y:.0f}/yr<extra></extra>"))
fig.add_trace(go.Scatter(x=100 * front["fill rate"], y=front["total $"], mode="lines+markers", name="frontier: cheapest for each fill rate",
                         line=dict(color="#4C78A8", width=3), customdata=front[["s", "S"]],
                         hovertemplate="s=%{customdata[0]}, S=%{customdata[1]}<br>fill %{x:.1f}%<br>$%{y:.0f}/yr<extra></extra>"))
fig.add_trace(go.Scatter(x=[100 * formula_avg["fill rate"]], y=[formula_avg["total $"]], mode="markers", name="formula policy (95% CSL)",
                         marker=dict(size=15, symbol="star", color="#E45756"),
                         hovertemplate="formula policy<br>fill %{x:.1f}%<br>$%{y:.0f}/yr<extra></extra>"))
fig.update_layout(title="Yearly cost vs fill rate for every (s, S) tried", xaxis_title="fill rate (%)",
                  yaxis_title="holding + ordering cost per year ($)", xaxis_range=[85, 100.5],
                  height=500, width=950, template="plotly_white", legend=dict(orientation="h", y=1.1))
fig.show()

rows = []
for target in [0.95, 0.98, 0.99, 0.995]:
    row = grid[grid["fill rate"] >= target].nsmallest(1, "total $").iloc[0]
    rows.append({"fill rate at least": f"{target:.1%}", "s": int(row.s), "S": int(row.S),
                 "fill rate": round(row["fill rate"], 3), "cost $/yr": round(row["total $"])})
pd.DataFrame(rows).set_index("fill rate at least")

253 policies simulated


,s,S,fill rate,cost $/yr
fill rate at least,,,,
95.0%,22,200,0.956,371
98.0%,34,180,0.987,391
99.0%,38,180,0.993,399
99.5%,38,220,0.996,410


**How to talk through this chart (it's the Day 3 "done when"):**
- **Each grey dot is one policy.** Dots high up are wasteful: there's a cheaper policy with the same fill rate.
- **The blue line is the frontier,** the cheapest way to reach each fill rate. Only policies on it are worth considering.
- **The frontier bends upward sharply at the right.** 95% fill costs about $371 a year, 99% about $399, and 99.5% about $410. **Each extra point of service costs more than the last.** The final half-point from 99% to 99.5% costs about $11, about 40% as much as the whole four-point step from 95% to 99% (about $28).
- **The formula policy (star) sits close to the frontier,** at about 99.9% fill for $421. A good start, slightly over-serving. Simulation tells you how to trim it.

**What you tell a planner:** *"Here's the menu. Pick a service level, and I'll tell you the cheapest policy and what it costs. If 99% is enough, we save about $22 a year on this item compared with the formula policy. Across thousands of SKUs, that adds up."*

## 6. When the lead time isn't fixed

Real suppliers aren't punctual. Suppose the lead time can be **2, 3, 4, 5, 6, 8 or 10 days**, equally likely: an average of 5.4 days, with a long tail.

The full safety stock formula has a **second term** for this:

```latex
SS = z \sqrt{L\,\sigma_d^2 + \mu_d^2\,\sigma_L^2}
```

- The first part, L σ_d², is the **demand** varying during an average lead time, as before.
- The new part, μ_d² σ_L², is the **lead time** varying. If the truck is 3 days late, you need 3 more days of average demand (3 × 8 = 24 units) whatever demand does.

What happens if we ignore that second term?

In [8]:
lead_options = np.array([2, 3, 4, 5, 6, 8, 10])
sampler = lambda rng: int(rng.choice(lead_options))
lead_mean, lead_sd = lead_options.mean(), lead_options.std()
print(f"lead time: mean {lead_mean:.2f} days, standard deviation {lead_sd:.2f} days")

rows = []
for name, sd_used in [("safety stock ignores lead-time spread", 0.0), ("safety stock includes it", lead_sd)]:
    pol = formula_policy(mu, sigma, lead_mean=lead_mean, lead_sd=sd_used)
    runs = pd.DataFrame([simulate(pol["s"], pol["S"], resampled_year(k), lead_time_sampler=sampler, seed=k)
                         for k in range(200)])
    rows.append({"policy": name, "safety stock": round(pol["safety stock"]), "s": pol["s"], "S": pol["S"],
                 "fill rate": runs["fill rate"].mean(), "cycle service level": runs["cycle service level"].mean(),
                 "total $": runs["total $"].mean()})
lead = pd.DataFrame(rows).set_index("policy")

fig = go.Figure()
for col in ["cycle service level", "fill rate"]:
    fig.add_trace(go.Bar(x=lead.index, y=100 * lead[col], name=col, hovertemplate="%{x}<br>%{y:.1f}%<extra></extra>"))
fig.add_hline(y=95, line_dash="dot", line_color="gray", annotation_text="95% CSL target")
fig.update_layout(title="Variable lead time (2-10 days), 95% CSL target, 200 simulated years", barmode="group",
                  yaxis_title="%", yaxis_range=[60, 100.5], height=430, width=900, template="plotly_white",
                  legend=dict(orientation="h", y=1.1))
fig.show()
lead.round(3)

lead time: mean 5.43 days, standard deviation 2.61 days


,safety stock,s,S,fill rate,cycle service level,total $
policy,,,,,,
safety stock ignores lead-time spread,17,60,213,0.975,0.762,426.549
safety stock includes it,38,81,234,0.993,0.911,478.151


**Ignoring lead-time variation** gives a safety stock of only 17 units. The CSL drops to **76%** against the 95% target, and the fill rate falls to **97.5%**. Late trucks cause most of the stockouts.

**Including it** more than doubles the safety stock (38 units). CSL rises to **91%** and fill rate to **99.3%**, for about $52 a year more.

It still falls short of 95%, and that's worth knowing. The formula assumes a bell-shaped lead time, but this one has a long tail (8 and 10 days). Orders can also overtake each other. **Formulas get you close; simulation tells you the truth.**

**Interview line (from the question bank):** *"Lead time from Vietnam just doubled. What changes?"* Expected demand over the lead time doubles, so s rises by μ × ΔL. If the lead time also got less reliable, the μ_d² σ_L² term grows fast, and often dominates the safety stock. Tell the business: *more stock is needed to keep the same service, roughly this many units and dollars; or accept lower service; or work on supplier reliability, which is often the cheapest lever.*

## Recap: say these out loud

1. **What's an (s, S) policy?** When inventory position (on hand + on order) falls to s, order up to S. s protects against stockouts during the lead time; S − s sets order size and frequency.
2. **How do you set s?** Average demand over the lead time + safety stock. SS = z σ_d √L, or z √(L σ_d² + μ_d² σ_L²) when lead times vary.
3. **How do you set S − s?** EOQ = √(2DK/h): it balances ordering cost against holding cost.
4. **Service level vs fill rate?** CSL = share of cycles with no stockout. Fill rate = share of demand served. Fill rate is usually much higher (99.4% at 80% CSL here), and it's what planners usually care about. Always ask which one a target means.
5. **The cost vs service curve:** it bends upward. Each extra point of fill rate costs more than the last.
6. **How does forecast error flow into safety stock?** Safety stock scales with the standard deviation of forecast error over the lead time. Better forecasts mean a smaller σ, less safety stock and lower cost at the same service. That's the dollar value of better forecasting.

**Words from today**

| Word | Meaning |
|---|---|
| Lead time | Days from placing an order to receiving it |
| Inventory position | On hand + on order (− backorders, if any) |
| Reorder point, s | Order when inventory position falls to this |
| Order-up-to level, S | Order enough to bring inventory position back to this |
| Safety stock | Buffer above average lead-time demand, to absorb variability |
| EOQ | Order size that balances ordering and holding cost |
| Cycle service level | Share of replenishment cycles with no stockout |
| Fill rate | Share of demand served from stock |
| Lost sales vs backorders | Unmet demand disappears, vs waits to be filled later |
| Efficient frontier | The cheapest policy for each service level |

**Try it yourself:**
1. Set `ORDER_COST = 40`. *Predict:* EOQ goes up by how much? (Hint: square root.) What happens to the fill rate at the same CSL?
2. Set `LEAD_TIME = 8`. *Predict:* how much do s and the yearly cost go up?